In [2]:
import os
import pandas as pd
from datetime import datetime
import pydicom
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import csv
import re


from utils.selection import *


# Source of RAW DICOMs
BERN_PATH='/nasdata/Berner_Daten/Other_Side_Bilddaten' 

DATA_PATH='/home/bule/berner_daten_analyse/data'
file_path =os.path.join(DATA_PATH, 'clinical','Kopie Olea von Relevent Data Other File 20181005.xlsx')

data = pd.read_excel(file_path)
#remove the last empty rows -> these are just commentings
data = data.iloc[:578] 
data.tail()

,Nr.,2012-2015 / 2015-2017,Fall Nr,PID,Datum Stroke,Alter (Jahre),Sex 0=Mann 1=Frau,Selbständig vor Stroke: mRS 0-2: 0=nein 1=ja,NIHSS bei Eintritt,NIHSS nach 24 h,...,Ort 2 VA 0=nein 1 =ja,Ort 2 PCA P1 0=nein 1=ja,Ort 2 PCA P2 0=nein 1=ja,SymOnset 1=sicher 2=zuletzt normal 3=wake-up-stroke 4=unklar,Lyse 0=nein 1=ja,"LyseTyp 1=iv 2=ia 3=iv,ia 4=iv und Sono 5=nur mech 6= als bridging geplant (iv) 7=keine",Konventionelle Angiographie 0=nein 1=ja,"Onset to treatment time (Lyse (2015-2017, IA oder Mechanisch 2012-2015)",Zeit Symptombeginn bis Angio [min] / Onset to Groin Puncture,Zeit Symptombeginn bis mechanisch / Stroke to reperfusion time
573,574.0,2015-2017,6.895914e+06,14273217.0,2017-08-27,79.3,1.0,1,13.0,0,...,0.0,0.0,0.0,5.0,1,1.0,1.0,120.0,233.0,NaN
574,575.0,2015-2017,6.901277e+06,11480130.0,2017-08-28,73.1,1.0,1,9.0,2,...,0.0,0.0,0.0,1.0,1,1.0,1.0,125.0,140.0,NaN
575,576.0,2015-2017,6.906639e+06,4864220.0,2017-08-28,77.6,1.0,1,15.0,9,...,0.0,0.0,0.0,3.0,0,7.0,1.0,NaN,844.0,NaN
576,577.0,2015-2017,6.912002e+06,6752837.0,2017-09-03,70.7,0.0,NaN,9.0,1,...,0.0,0.0,0.0,1.0,1,1.0,1.0,95.0,174.0,NaN
577,578.0,2015-2017,6.917364e+06,2679663.0,2017-09-07,33.6,1.0,1,12.0,0,...,0.0,0.0,0.0,5.0,0,7.0,1.0,NaN,620.0,NaN


check stroke date and image date

In [ ]:
data.info() # -> open in data wrangler  

<class 'pandas.core.frame.DataFrame'>
Index: 575 entries, 0 to 577
Data columns (total 55 columns):
 #   Column                                                                                                                                                                  Non-Null Count  Dtype         
---  ------                                                                                                                                                                  --------------  -----         
 0   Nr.                                                                                                                                                                     575 non-null    float64       
 1   2012-2015 / 2015-2017                                                                                                                                                   575 non-null    object        
 2   Fall Nr                                                                        

In [7]:
data

,Nr.,2012-2015 / 2015-2017,Fall Nr,PID,Datum Stroke,Alter (Jahre),Sex 0=Mann 1=Frau,Selbständig vor Stroke: mRS 0-2: 0=nein 1=ja,NIHSS bei Eintritt,NIHSS nach 24 h,...,Ort 2 VA 0=nein 1 =ja,Ort 2 PCA P1 0=nein 1=ja,Ort 2 PCA P2 0=nein 1=ja,SymOnset 1=sicher 2=zuletzt normal 3=wake-up-stroke 4=unklar,Lyse 0=nein 1=ja,"LyseTyp 1=iv 2=ia 3=iv,ia 4=iv und Sono 5=nur mech 6= als bridging geplant (iv) 7=keine",Konventionelle Angiographie 0=nein 1=ja,"Onset to treatment time (Lyse (2015-2017, IA oder Mechanisch 2012-2015)",Zeit Symptombeginn bis Angio [min] / Onset to Groin Puncture,Zeit Symptombeginn bis mechanisch / Stroke to reperfusion time
0,1.0,2012-2015,3.817992e+06,10792082.0,2012-01-02,82.865753,1.0,1,16.0,NaN,...,0.0,0.0,0.0,2.0,0,5.0,1.0,515.0,487.0,515.0
1,2.0,2012-2015,3.836042e+06,10822127.0,2012-01-21,61.016438,0.0,1,12.0,NaN,...,0.0,0.0,0.0,2.0,1,3.0,1.0,330.0,307.0,330.0
2,3.0,2012-2015,3.836174e+06,10823107.0,2012-01-21,50.775342,0.0,1,13.0,10,...,0.0,0.0,0.0,1.0,1,3.0,1.0,255.0,246.0,255.0
3,4.0,2012-2015,3.842328e+06,82691045.0,2012-01-28,65.416438,1.0,1,23.0,36,...,0.0,0.0,0.0,1.0,0,5.0,1.0,240.0,207.0,240.0
4,5.0,2012-2015,3.848312e+06,10841989.0,2012-02-03,61.967123,1.0,1,15.0,4,...,0.0,0.0,0.0,1.0,1,3.0,1.0,330.0,154.0,330.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
573,574.0,2015-2017,6.895914e+06,14273217.0,2017-08-27,79.300000,1.0,1,13.0,0,...,0.0,0.0,0.0,5.0,1,1.0,1.0,120.0,233.0,NaN
574,575.0,2015-2017,6.901277e+06,11480130.0,2017-08-28,73.100000,1.0,1,9.0,2,...,0.0,0.0,0.0,1.0,1,1.0,1.0,125.0,140.0,NaN
575,576.0,2015-2017,6.906639e+06,4864220.0,2017-08-28,77.600000,1.0,1,15.0,9,...,0.0,0.0,0.0,3.0,0,7.0,1.0,NaN,844.0,NaN
576,577.0,2015-2017,6.912002e+06,6752837.0,2017-09-03,70.700000,0.0,NaN,9.0,1,...,0.0,0.0,0.0,1.0,1,1.0,1.0,95.0,174.0,NaN


# Klinische Daten Lisa

In [3]:

file_path = '/home/bule/berner_daten_analyse/data/clinical/lisa_stroke/data_wide_all_nihss.csv'

data_lisa = pd.read_csv(file_path)

for col in data_lisa.columns:
    print(col)

p_id
age
sex
independent_pre_stroke
nihss_bl
nihss_24h
nihss_diff_bl_24h
intracranial_bleeding
tah_pre_stroke
antikoagulation_pre_stroke
statin_pre_stroke
antihypertensiva_pre_stroke
sys_bloodpressure_bl
dias_bloodpressure_bl
glucose_bl
hba1c
ldl
hdl
triglyceride
crp
inr
anaesthesia
atrial_fibrillation
rf_diabetes
rf_hypertonia
rf_hypercholesterinaemie
rf_smoker
rf_chd
rf_pavk
rf_tia_stroke
infarct_side
additional_occlusion
lyse
time_to_imaging
time_to_groin_puncture
vessel_open_before_therapy
tici
tici_binary
volume_adc
volume_tmax
volume_rbf_adc
volume_rbf_tmax
volume_rbv_adc
volume_rbv_tmax
volume_tar
mrs_3months_binary
collateralization
C_Ant_mtt
C_Ant_rbf
C_Ant_rbv
C_Ant_tmax
C_Ant_tmip
C_Ant_ttp
C_LN_mtt
C_LN_rbf
C_LN_rbv
C_LN_tmax
C_LN_tmip
C_LN_ttp
C_Meda_mtt
C_Meda_rbf
C_Meda_rbv
C_Meda_tmax
C_Meda_tmip
C_Meda_ttp
C_Medm_mtt
C_Medm_rbf
C_Medm_rbv
C_Medm_tmax
C_Medm_tmip
C_Medm_ttp
C_Medp_mtt
C_Medp_rbf
C_Medp_rbv
C_Medp_tmax
C_Medp_tmip
C_Medp_ttp
C_Occ_mtt
C_Occ_rbf
C_Occ_rbv

In [8]:
data_lisa['mrs_3months_binary']=data_lisa['mrs_3months_binary'].astype('category')
data_lisa['mrs_3months_binary'].value_counts()

mrs_3months_binary
1    119
0    103
Name: count, dtype: int64

In [ ]:
103/(222) ~ 46% of the patients have a good outcome

0.46396396396396394

# Imputation

In [ ]:
import pandas as pd
import numpy as np
from tabpfn import TabPFNClassifier

data_encoded = pd.get_dummies(data, dummy_na=True)

# Impute missing values using TabPFN
for column in data.columns:
    if data[column].isnull().any():
        print(f"Imputing column: {column}")
        
        target = data[column]
        features = data.drop(columns=[column])
        mask = target.notnull()
        
        clf = TabPFNClassifier(device='cpu')  # Use 'cuda' if GPU is available
        clf.fit(features[mask], target[mask])
        
        missing_mask = ~mask
        predictions = clf.predict(features[missing_mask])
        
        data.loc[missing_mask, column] = predictions

print(data)